# LLM File Formats

A theory-only reference on the file formats used to store, distribute, and deploy LLM
weights — what each one is, how it works internally, and when it's used.

**Contents**
1. Training / research formats (PyTorch-native)
2. Deployment / inference-optimized formats
3. Quantized weight formats
4. Quick reference: which format for which situation


## 1. Training / Research Formats (PyTorch-native)

These are the formats models are usually saved in during and immediately after
training, before any deployment-specific optimization happens.

### `.bin` / `.pt` / `.pth`
Raw PyTorch state dicts, saved via `torch.save()`. Internally these use Python's
**pickle** protocol to serialize the tensor data and metadata together.

- **Security risk**: pickle can execute arbitrary code on load. Loading a `.bin`/`.pt`
  file from an untrusted source is a genuine code-execution risk, not just a
  theoretical one.
- Historically the default format on the Hugging Face Hub (`pytorch_model.bin`), and
  still very common, but increasingly being replaced by `.safetensors`.

### `.safetensors`
Developed by Hugging Face specifically to fix the pickle problem above.

- Structure: a JSON header describing tensor names, shapes, and dtypes, followed
  directly by the raw tensor bytes — no arbitrary code execution possible on load.
- Supports **zero-copy, memory-mapped loading**, which often makes it faster to load
  than `.bin` in addition to being safer.
- Now the default format on the Hugging Face Hub.

### `.ckpt`
Checkpoint files, common in the Stable Diffusion / PyTorch Lightning ecosystem.

- Also pickle-based internally, carrying the same security caveat as `.bin`.
- Usually bundles more than just weights — optimizer state, epoch/step number,
  learning-rate scheduler state, etc. — so files tend to be larger than pure-weight
  formats.

### `.h5`
Keras/TensorFlow's HDF5-based format, produced by `model.save()`.

- Rare for modern large language models, since most major open LLMs are released as
  PyTorch or safetensors, but still seen with older TensorFlow-based models.


## 2. Deployment / Inference-Optimized Formats

These formats exist to make trained weights run efficiently on specific hardware or
runtimes, often trading some portability for speed or footprint.

### GGUF (successor to GGML)
Used by `llama.cpp` and the tools built on it (Ollama, LM Studio, koboldcpp, etc.).

- A single, self-contained file: model weights, metadata, and the tokenizer are all
  bundled together.
- Purpose-built for efficient **CPU and consumer-GPU inference**.
- Carries the various quantization levels (e.g. Q4_K_M, Q5_K_S, Q8_0) that trade file
  size and speed against accuracy — this is the format most local-inference tools
  expect.

### ONNX (`.onnx`)
An open, framework-agnostic computation-graph format.

- Models trained in PyTorch or TensorFlow can be exported to ONNX and then run
  anywhere an ONNX Runtime exists — mobile devices, browsers (via WebAssembly), edge
  hardware.
- More commonly used for smaller/encoder models and embedding models than for huge
  decoder-only LLMs, though large-LLM ONNX exports do exist.

### TensorRT Engines (`.engine` / `.plan`)
NVIDIA-specific compiled inference artifacts.

- Optimized and compiled for a **particular GPU architecture**.
- Gives the fastest possible inference on the target NVIDIA hardware, but is **not
  portable** — an engine compiled for one GPU generation (e.g. Ampere) generally
  won't run on a different generation without recompiling.

### MLX
Apple's format/framework for Apple Silicon.

- Optimized specifically for the **unified memory architecture** of M-series chips.
- The format used when running models locally on a Mac via the `mlx-lm` ecosystem.


## 3. Quantized Weight Formats

These aren't file extensions in their own right so much as **quantization schemes**
that produce weights typically packaged inside one of the formats above (most often
GGUF, but also standalone).

### GPTQ
- Post-training quantization, typically down to 4-bit precision.
- GPU-focused.
- Was popular before GGUF became the dominant choice for consumer-hardware inference.

### AWQ (Activation-aware Quantization)
- Also GPU-focused.
- Generally achieves better accuracy retention than GPTQ at similar bit-widths, by
  accounting for which weights are most "activation-sensitive" when deciding how
  aggressively to quantize them.

### GGUF Quantization
- The quantization levels bundled inside GGUF files themselves (Q2 through Q8, with
  variants like `_K_M`/`_K_S` indicating different mixed-precision strategies).
- The path most associated with running models on CPU or consumer-grade GPUs.


## 4. Quick Reference: Which Format for Which Situation

| Situation | Format |
|---|---|
| Downloading or fine-tuning a model from Hugging Face | `.safetensors` |
| Running locally on CPU, or on a Mac via Ollama / LM Studio | GGUF |
| Deploying cross-platform (mobile, browser, edge) | ONNX |
| Maximum throughput on a specific NVIDIA GPU in production | TensorRT |
| Inherited an old repo with `.bin` or `.ckpt` files | Convert to `.safetensors` where possible; never unpickle a file from an untrusted source |

**Key takeaways**
- `.bin`, `.pt`, `.pth`, and `.ckpt` all rely on pickle internally, which makes them a
  security risk when the source isn't trusted.
- `.safetensors` is the modern default for storing and sharing weights: safe to load
  and often faster.
- GGUF, ONNX, TensorRT, and MLX are all deployment-oriented formats, each tied to a
  different target environment (consumer CPU/GPU, cross-platform, NVIDIA-specific,
  and Apple Silicon respectively) rather than being interchangeable.
- GPTQ, AWQ, and GGUF's own quantization levels are schemes for shrinking model size
  and speeding up inference, usually at some cost to accuracy.
